# Behaviour of algorithms in a 6-node network

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import arviz as az
from scipy.special import logsumexp, betaln, gammaln
from itertools import combinations
from time import perf_counter

import sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from source.SBM import SBMModel
from source.PartitionPrior import PartitionPrior
from source.Sampler import BaseSampler, SamplerConfig, GibbsSampler, SplitMergeSampler, ChaperonesSampler

In [6]:
def canonical_partition(z):
    """
    Convert a partition into its canonical representation.
    """
    mapping = {}
    result = []

    for label in z:
        if label not in mapping:
            mapping[label] = len(mapping)
        result.append(mapping[label])

    return tuple(result)

In [7]:
Y6 = np.array([
    [0, 1, 1, 0, 0, 0],
    [1, 0, 1, 0, 0, 0],
    [1, 1, 0, 1, 0, 0],
    [0, 0, 1, 0, 1, 1],
    [0, 0, 0, 1, 0, 0],
    [0, 0, 0, 1, 0, 0],
    ], dtype=int)

a = 1.0
b = 1.0
alpha = 1.5
assert Y6.shape == (6, 6)
assert np.array_equal(Y6, Y6.T)
assert np.all(np.diag(Y6) == 0)
assert np.all(np.isin(Y6, [0, 1]))
print("Number of nodes:", len(Y6))
print("Number of edges:", int(np.triu(Y6, k=1).sum()))


def enumerate_partitions(n):
    """Enumerates all partitions of n elements."""
    if n < 1:
        raise ValueError("n must be positive.")
    partitions = []
    def recurse(z, max_label):
        i = len(z)
        if i == n:
            partitions.append(tuple(z))
            return
        for label in range(max_label + 2):
            recurse(z + [label], max(max_label, label))
    recurse([0], 0)
    return partitions


partitions = enumerate_partitions(6)

print("Partitions enumerated:", len(partitions))


def log_marginal_sbm(Y, z, a=1.0, b=1.0):
    """Log-likelihood marginal of a binary undirected SBM."""
    Y = np.asarray(Y)
    z = np.asarray(z)
    n = len(z)
    if Y.shape != (n, n):
        raise ValueError("Incompatible dimensions between Y and z.")
    if not np.array_equal(Y, Y.T):
        raise ValueError("Y must be symmetric.")
    if np.any(np.diag(Y) != 0):
        raise ValueError("Y must have a zero diagonal.")
    if not np.all(np.isin(Y, [0, 1])):
        raise ValueError("Y must be binary.")
    if a <= 0 or b <= 0:
        raise ValueError("a and b must be positive.")
    labels = sorted(set(z))
    total = 0.0

    for pos, h in enumerate(labels):
        for k in labels[pos:]:
            nodes_h = np.flatnonzero(z == h)
            nodes_k = np.flatnonzero(z == k)
            if h == k:
                possible = len(nodes_h) * (len(nodes_h) - 1) // 2
                subgraph = Y[np.ix_(nodes_h, nodes_h)]
                edges = int(np.triu(subgraph, k=1).sum())
            else:
                possible = len(nodes_h) * len(nodes_k)
                edges = int(Y[np.ix_(nodes_h, nodes_k)].sum())

            non_edges = possible - edges

            total += betaln(a + edges, b + non_edges) - betaln(a, b)
    return float(total)

def log_prior_dp(z, alpha=1.5):
    z = np.asarray(z)
    n = len(z)
    sizes = np.unique(z, return_counts=True)[1]
    K = len(sizes)
    return gammaln(alpha) - gammaln(alpha + n) + K * np.log(alpha) + np.sum(gammaln(sizes)) 


log_priors = np.array([log_prior_dp(z, alpha) for z in partitions])
prior_probabilities = np.exp(log_priors)
print("Sum of prior probabilities:", prior_probabilities.sum())
assert np.isclose(prior_probabilities.sum(), 1.0)
log_likelihoods = np.array([log_marginal_sbm(Y6, z, a, b) for z in partitions])
log_joint = log_likelihoods + log_priors
log_evidence = logsumexp(log_joint)
exact_probabilities = np.exp(log_joint - log_evidence)
print("Sum of posterior probabilities:", exact_probabilities.sum())
assert np.isclose(exact_probabilities.sum(), 1.0)
order = np.argsort(exact_probabilities)[::-1]

print("\n Fifteen partitions with highest posterior probability:")
for idx in order[:15]:
    print(f"z={partitions[idx]}, "
        f"K={len(set(partitions[idx]))}, "
        f"p(z|Y)={exact_probabilities[idx]:.6f}" )


model4 = SBMModel(adjacency=Y6, a=a, b=b)
prior4 = PartitionPrior(kind="DP", alpha=alpha)
config4 = SamplerConfig(n_iterations=55_000, burn_in=5_000,thin=1)

Number of nodes: 6
Number of edges: 6
Partitions enumerated: 203
Sum of prior probabilities: 0.9999999999999998
Sum of posterior probabilities: 0.9999999999999997

 Fifteen partitions with highest posterior probability:
z=(0, 0, 0, 1, 2, 2), K=3, p(z|Y)=0.076930
z=(0, 0, 0, 0, 0, 0), K=1, p(z|Y)=0.051646
z=(0, 0, 0, 1, 1, 1), K=2, p(z|Y)=0.047868
z=(0, 0, 0, 1, 2, 3), K=4, p(z|Y)=0.037864
z=(0, 0, 1, 2, 3, 3), K=4, p(z|Y)=0.035901
z=(0, 0, 1, 2, 2, 2), K=3, p(z|Y)=0.025643
z=(0, 0, 0, 1, 2, 1), K=3, p(z|Y)=0.019233
z=(0, 0, 0, 1, 1, 2), K=3, p(z|Y)=0.019233
z=(0, 1, 1, 2, 3, 3), K=4, p(z|Y)=0.017950
z=(0, 1, 0, 2, 3, 3), K=4, p(z|Y)=0.017950
z=(0, 1, 2, 3, 4, 4), K=5, p(z|Y)=0.016829
z=(0, 0, 1, 2, 3, 4), K=5, p(z|Y)=0.016829
z=(0, 0, 1, 0, 0, 0), K=2, p(z|Y)=0.015666
z=(0, 0, 0, 1, 0, 0), K=2, p(z|Y)=0.015666
z=(0, 0, 1, 2, 1, 1), K=3, p(z|Y)=0.015386


In [8]:
def tv_distance_trajectory(
    chains,
    partitions,
    exact_probabilities
):
    """
    Calculate the cumulative total variation distance
    across multiple independent chains.

    At each iteration, all chains contribute the same
    number of samples.
    """

    n_chains = len(chains)
    n_iterations = min(len(chain) for chain in chains)

    partition_index = {
        z: i for i, z in enumerate(partitions)
    }

    exact = np.asarray(exact_probabilities)

    indices = np.array([
        [
            partition_index[canonical_partition(z)]
            for z in chain[:n_iterations]
        ]
        for chain in chains
    ])

    n_states = len(partitions)

    counts = np.zeros(n_states, dtype=int)
    tv = np.zeros(n_iterations)

    for t in range(n_iterations):

        for c in range(n_chains):
            counts[indices[c, t]] += 1

        empirical = counts / ((t + 1) * n_chains)

        tv[t] = 0.5 * np.sum(
            np.abs(empirical - exact)
        )

    return tv

In [9]:
def build_diagnostic_variables(chains, partitions, exact_probabilities, top_m=10):
    """
    Build diagnostic variables for each chain:

    - Number of communities (K).
    - Pairwise co-clustering indicators.
    - Indicators of the most relevant partitions.
    """

    n_chains = len(chains)
    n_iterations = min(len(chain) for chain in chains)

    n_nodes = len(chains[0][0])

    top_indices = np.argsort(
        exact_probabilities
    )[::-1][:top_m]

    top_partitions = [
        partitions[i] for i in top_indices
    ]

    variables = {}

    for c, chain in enumerate(chains):

        chain = chain[:n_iterations]

        # Number of communities
        K = np.array([
            len(set(z)) for z in chain
        ])

        variables.setdefault("K", []).append(K)

        # Pairwise co-clustering indicators
        for i, j in combinations(range(n_nodes), 2):

            pair_indicator = np.array([
                int(z[i] == z[j])
                for z in chain
            ])

            name = f"pair_{i}_{j}"

            variables.setdefault(name, []).append(
                pair_indicator
            )

        # Indicators for the top posterior partitions
        for r, partition in enumerate(top_partitions):

            state_indicator = np.array([
                int(canonical_partition(z) == partition)
                for z in chain
            ])

            name = f"top_partition_{r+1}"

            variables.setdefault(name, []).append(
                state_indicator
            )

    return variables, top_partitions

In [10]:
def calculate_diagnostics(
    chains,
    partitions,
    exact_probabilities,
    top_m=10
):
    """
    Calculate ESS and R-hat across independent chains.
    """

    variables, top_partitions = build_diagnostic_variables(
        chains,
        partitions,
        exact_probabilities,
        top_m=top_m
    )

    # Convert diagnostic variables into an ArviZ InferenceData object
    posterior = {
        name: np.asarray(chain_values)
        for name, chain_values in variables.items()
    }

    idata = az.from_dict(posterior=posterior)

    # Bulk ESS and rank-normalized R-hat
    ess_result = az.ess(
        idata,
        method="bulk"
    )

    rhat_result = az.rhat(
        idata,
        method="rank"
    )

    ess_values = {
        name: float(np.asarray(ess_result[name]).item())
        for name in ess_result.data_vars
    }

    rhat_values = {
        name: float(np.asarray(rhat_result[name]).item())
        for name in rhat_result.data_vars
    }

    # Organize ESS by diagnostic category
    ess_K = ess_values["K"]

    ess_pairs = {
        name: value
        for name, value in ess_values.items()
        if name.startswith("pair_")
    }

    ess_top = {
        name: value
        for name, value in ess_values.items()
        if name.startswith("top_partition_")
    }

    # Exclude non-finite values from summaries
    valid_ess = np.array([
        value for value in ess_values.values()
        if np.isfinite(value)
    ])

    valid_rhat = np.array([
        value for value in rhat_values.values()
        if np.isfinite(value)
    ])

    return {
        "ESS_K": ess_K,

        "ESS_pairs": ess_pairs,
        "ESS_pairs_mean": np.mean(list(ess_pairs.values())),
        "ESS_pairs_min": np.min(list(ess_pairs.values())),
        "ESS_pairs_median": np.median(list(ess_pairs.values())),

        "ESS_top_partitions": ess_top,
        "ESS_top_mean": np.mean(list(ess_top.values())),

        "ESS_mean": np.mean(valid_ess),
        "ESS_min": np.min(valid_ess),
        "ESS_median": np.median(valid_ess),

        "Rhat": rhat_values,
        "Rhat_max": np.max(valid_rhat),
        "Rhat_K": rhat_values["K"],
        "Rhat_mean": np.mean(valid_rhat),

        "all_ESS": ess_values,
        "top_partitions": top_partitions
    }

In [11]:
def run_multiple_chains(
    sampler_class,
    model,
    prior,
    config,
    initial_partition,
    seeds
):
    """
    Run multiple independent chains and measure
    the computational time.
    """

    chains = []
    execution_times = []

    for seed in seeds:

        sampler = sampler_class(
            model=model,
            prior=prior,
            config=config
        )

        start = perf_counter()

        result = sampler.fit(
            initial_partition=np.array(initial_partition),
            seed=seed
        )

        elapsed = perf_counter() - start

        raw_samples = result["samples"]

        samples = [
            canonical_partition(z)
            for z in raw_samples
        ]

        chains.append(samples)
        execution_times.append(elapsed)

        print(
            f"Seed {seed}: "
            f"{elapsed:.3f} seconds | "
            f"{len(samples)} retained samples"
        )

    return {
        "chains": chains,
        "execution_times": execution_times,
        "total_time": sum(execution_times),
        "mean_time": np.mean(execution_times),
        "n_chains": len(chains)
    }

In [12]:
def evaluate_sampler(
    sampler_class,
    name,
    model,
    prior,
    config,
    initial_partition,
    seeds,
    partitions,
    exact_probabilities,
    top_m=10
):
    """
    Complete evaluation of a Bayesian partition sampler.
    """

    print(f"\n{'='*60}")
    print(f"Evaluating: {name}")
    print(f"{'='*60}")

    # Run independent chains and measure execution time
    execution = run_multiple_chains(
        sampler_class=sampler_class,
        model=model,
        prior=prior,
        config=config,
        initial_partition=initial_partition,
        seeds=seeds
    )

    chains = execution["chains"]

    # Total variation distance
    tv = tv_distance_trajectory(
        chains,
        partitions,
        exact_probabilities
    )

    # ESS and R-hat diagnostics
    diagnostics = calculate_diagnostics(
        chains,
        partitions,
        exact_probabilities,
        top_m=top_m
    )

    total_samples = sum(len(chain) for chain in chains)

    total_time = execution["total_time"]

    # ESS per second
    ess_per_second = (
        diagnostics["ESS_K"] / total_time
    )

    print(f"\nNumber of chains: {execution['n_chains']}")
    print(f"Total retained samples: {total_samples}")
    print(f"Total computational time: {total_time:.3f} seconds")
    print(f"Mean time per chain: {execution['mean_time']:.3f} seconds")
    print(f"Final TV distance: {tv[-1]:.6f}")

    print(f"ESS of K: {diagnostics['ESS_K']:.2f}")
    print(f"ESS of pairwise co-clustering (mean): "
          f"{diagnostics['ESS_pairs_mean']:.2f}")
    print(f"ESS of top partitions (mean): "
          f"{diagnostics['ESS_top_mean']:.2f}")

    print(f"Minimum ESS: {diagnostics['ESS_min']:.2f}")
    print(f"Median ESS: {diagnostics['ESS_median']:.2f}")

    print(f"Maximum R-hat: {diagnostics['Rhat_max']:.4f}")
    print(f"R-hat of K: {diagnostics['Rhat_K']:.4f}")

    print(f"ESS of K per second: {ess_per_second:.2f}")

    return {
        "name": name,
        "chains": chains,
        "TV": tv,
        "diagnostics": diagnostics,
        "execution": execution,
        "total_samples": total_samples,
        "ESS_per_second": ess_per_second
    }

In [13]:
# Sampling configuration
config4 = SamplerConfig(
    n_iterations=15_000,
    burn_in=5_000,
    thin=1
)

# Independent random seeds
seeds = [123, 456, 789, 2026]

# Initial partition
initial_partition = np.array([0, 0, 1, 1, 1, 1])

# Run and evaluate Gibbs
results = {}

results["Gibbs"] = evaluate_sampler(
    sampler_class=GibbsSampler,
    name="Gibbs",
    model=model4,
    prior=prior4,
    config=config4,
    initial_partition=initial_partition,
    seeds=seeds,
    partitions=partitions,
    exact_probabilities=exact_probabilities,
    top_m=10
)

# Run and evaluate Split-Merge
results["Split-Merge"] = evaluate_sampler(
    sampler_class=SplitMergeSampler,
    name="Split-Merge",
    model=model4,
    prior=prior4,
    config=config4,
    initial_partition=initial_partition,
    seeds=seeds,
    partitions=partitions,
    exact_probabilities=exact_probabilities,
    top_m=10
)

# Run and evaluate Chaperones
results["Chaperones"] = evaluate_sampler(
    sampler_class=ChaperonesSampler,
    name="Chaperones",
    model=model4,
    prior=prior4,
    config=config4,
    initial_partition=initial_partition,
    seeds=seeds,
    partitions=partitions,
    exact_probabilities=exact_probabilities,
    top_m=10
)


Evaluating: Gibbs


Sampling: 100%|██████████| 15000/15000 [02:57<00:00, 84.40it/s] 


Seed 123: 177.767 seconds | 10000 retained samples


Sampling: 100%|██████████| 15000/15000 [01:59<00:00, 125.01it/s]


Seed 456: 120.004 seconds | 10000 retained samples


Sampling: 100%|██████████| 15000/15000 [01:34<00:00, 158.66it/s]


Seed 789: 94.556 seconds | 10000 retained samples


Sampling: 100%|██████████| 15000/15000 [02:01<00:00, 123.78it/s]


Seed 2026: 121.193 seconds | 10000 retained samples


TypeError: from_dict() got an unexpected keyword argument 'posterior'

In [ ]:
plt.figure(figsize=(11, 6))

for name, res in results.items():

    plt.plot(
        np.arange(1, len(res["TV"]) + 1),
        res["TV"],
        label=name,
        linewidth=1.5
    )

plt.xscale("log")

plt.xlabel("Iterations per chain")
plt.ylabel("Total variation distance")

plt.title("Convergence towards the exact posterior distribution")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
time_comparison = pd.DataFrame({
    name: {
        "Total time (s)": res["execution"]["total_time"],
        "Mean time per chain (s)": res["execution"]["mean_time"],
        "Time per retained sample (ms)": (
            1000 * res["execution"]["total_time"]
            / res["total_samples"]
        )
    }
    for name, res in results.items()
}).T

print(time_comparison)

time_comparison["Total time (s)"].plot(
    kind="bar",
    figsize=(9, 5)
)

plt.ylabel("Computational time (seconds)")
plt.title("Computational cost comparison")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()  

In [ ]:
ess_comparison = pd.DataFrame({
    name: {
        "ESS of K": res["diagnostics"]["ESS_K"],
        "ESS of pairwise co-clustering": res["diagnostics"]["ESS_pairs_mean"],
        "ESS of top partitions": res["diagnostics"]["ESS_top_mean"],
        "Minimum ESS": res["diagnostics"]["ESS_min"],
        "Median ESS": res["diagnostics"]["ESS_median"]
    }
    for name, res in results.items()
}).T

print(ess_comparison.round(2))


ess_comparison.plot(
    kind="bar",
    figsize=(12, 6),
    width=0.8
)

plt.ylabel("Effective Sample Size")
plt.title("ESS comparison across sampling algorithms")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
rhat_comparison = pd.DataFrame({
    name: {
        "R-hat of K": res["diagnostics"]["Rhat_K"],
        "Maximum R-hat": res["diagnostics"]["Rhat_max"],
        "Mean R-hat": res["diagnostics"]["Rhat_mean"]
    }
    for name, res in results.items()
}).T

print(rhat_comparison.round(4))

rhat_comparison.plot(
    kind="bar",
    figsize=(10, 5)
)

plt.axhline(
    y=1.01,
    linestyle="--",
    label="R-hat threshold (1.01)"
)

plt.ylabel("R-hat")
plt.title("Convergence diagnostics across independent chains")
plt.xticks(rotation=0)
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
pairwise_ess = pd.DataFrame({
    name: res["diagnostics"]["ESS_pairs"]
    for name, res in results.items()
}).T

print(pairwise_ess.round(2))

In [ ]:
efficiency_comparison = pd.DataFrame({
    name: {
        "ESS/s (K)": res["ESS_per_second"],
        "ESS/s (pairs)": (
            res["diagnostics"]["ESS_pairs_mean"]
            / res["execution"]["total_time"]
        ),
        "ESS/s (top partitions)": (
            res["diagnostics"]["ESS_top_mean"]
            / res["execution"]["total_time"]
        )
    }
    for name, res in results.items()
}).T

print(efficiency_comparison.round(2))

efficiency_comparison.plot(
    kind="bar",
    figsize=(10, 6)
)

plt.ylabel("Effective samples per second")
plt.title("Computational efficiency comparison")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
summary = pd.DataFrame({
    name: {
        "Final TV": res["TV"][-1],

        "ESS K": res["diagnostics"]["ESS_K"],

        "ESS pairs (mean)": (
            res["diagnostics"]["ESS_pairs_mean"]
        ),

        "ESS top partitions (mean)": (
            res["diagnostics"]["ESS_top_mean"]
        ),

        "ESS minimum": res["diagnostics"]["ESS_min"],

        "ESS median": res["diagnostics"]["ESS_median"],

        "Maximum R-hat": res["diagnostics"]["Rhat_max"],

        "Total time (s)": res["execution"]["total_time"],

        "ESS K per second": res["ESS_per_second"]
    }
    for name, res in results.items()
}).T

print("\nFINAL ALGORITHM COMPARISON")
print(summary.round(4))

summary.to_csv(
    "algorithm_comparison_results.csv",
    index=True
)